# 01 — Data Cleaning & Validation

**Project:** E-Commerce Customer & Revenue Analytics

**Business question this project answers:** What is driving revenue performance, which customers and products are most valuable, and where should the company focus to improve customer retention and revenue growth?

**Purpose of this notebook:** before any KPI, RFM, cohort, or product analysis can be trusted, the raw source tables need to be profiled, validated, and reconciled with each other. This notebook:

1. Loads the four raw source tables (`customers`, `products`, `orders`, `order_items`)
2. Profiles each table (schema, nulls, duplicates, ranges, categorical values)
3. Checks referential integrity **across** tables (orphan records in either direction)
4. Documents the cleaning decisions and business rules applied, with reasoning
5. Builds a single analysis-ready **`fact_sales`** table (order-line grain) used by every downstream notebook
6. Sanity-checks the rebuilt table against the analyst-provided `sales_cleaned.csv`
7. Writes the cleaned tables to `data/processed/` for reuse in SQL, later notebooks, and the Power BI model

**Source data:** `data/raw/` — `customers.csv`, `products.csv`, `orders.csv`, `order_items.csv`, and a reference file `sales_cleaned.csv` (a pre-joined extract, used here only as a sanity check, not as an input).

In [13]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_RAW = ROOT / "data" / "raw"
DATA_PROCESSED = ROOT / "data" / "processed"
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

print("Project root:", ROOT)
print("Raw data dir:", DATA_RAW)

Project root: /Users/yvonne/Desktop/E-commerce Analytics
Raw data dir: /Users/yvonne/Desktop/E-commerce Analytics/data/raw


## 1. Load raw data

In [14]:
customers = pd.read_csv(DATA_RAW / "customers.csv")
products = pd.read_csv(DATA_RAW / "products.csv")
orders = pd.read_csv(DATA_RAW / "orders.csv")
order_items = pd.read_csv(DATA_RAW / "order_items.csv")

for name, df in [("customers", customers), ("products", products),
                  ("orders", orders), ("order_items", order_items)]:
    print(f"{name:12s} shape={df.shape}")

customers    shape=(300, 3)
products     shape=(50, 3)
orders       shape=(1000, 4)
order_items  shape=(2000, 4)


## 2. Schema & dtype audit

Quick look at column types and the first few rows of each raw table before any cleaning.

In [15]:
for name, df in [("customers", customers), ("products", products),
                  ("orders", orders), ("order_items", order_items)]:
    print("=" * 60)
    print(name)
    print("=" * 60)
    df.info()
    display(df.head(3))

customers
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 300 entries, 0 to 299
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   customer_id  300 non-null    int64 
 1   country      300 non-null    object
 2   signup_date  300 non-null    object
dtypes: int64(1), object(2)
memory usage: 7.2+ KB


,customer_id,country,signup_date
0,1,Spain,2023-05-15
1,2,Italy,2023-02-10
2,3,Spain,2023-10-23


products
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   product_id    50 non-null     int64 
 1   product_name  50 non-null     object
 2   category      50 non-null     object
dtypes: int64(1), object(2)
memory usage: 1.3+ KB


,product_id,product_name,category
0,1,Product_1,Body
1,2,Product_2,Skin
2,3,Product_3,Hair


orders
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   order_id     1000 non-null   int64 
 1   customer_id  1000 non-null   int64 
 2   order_date   1000 non-null   object
 3   status       1000 non-null   object
dtypes: int64(2), object(2)
memory usage: 31.4+ KB


,order_id,customer_id,order_date,status
0,1,103,2024-10-31,Completed
1,2,271,2024-10-12,Completed
2,3,107,2024-05-23,Completed


order_items
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   order_id    2000 non-null   int64
 1   product_id  2000 non-null   int64
 2   quantity    2000 non-null   int64
 3   price       2000 non-null   int64
dtypes: int64(4)
memory usage: 62.6 KB


,order_id,product_id,quantity,price
0,508,48,3,108
1,290,18,2,93
2,295,16,2,27


## 3. `customers` — data quality checks

In [16]:
print("Rows:", len(customers))
print("Duplicate customer_id:", customers["customer_id"].duplicated().sum())
print("Fully duplicate rows:", customers.duplicated().sum())
print("\nNulls per column:\n", customers.isnull().sum())

customers["signup_date"] = pd.to_datetime(customers["signup_date"], errors="coerce")
print("\nUnparseable signup_date after coercion:", customers["signup_date"].isnull().sum())
print("signup_date range:", customers["signup_date"].min(), "to", customers["signup_date"].max())

print("\nCountry value counts:")
print(customers["country"].value_counts(dropna=False))

# whitespace / casing inconsistencies that would silently split a "country" group-by
stripped = customers["country"].astype(str).str.strip()
print("\nCountry values with leading/trailing whitespace:", (customers["country"].astype(str) != stripped).sum())

Rows: 300
Duplicate customer_id: 0
Fully duplicate rows: 0

Nulls per column:
 customer_id    0
country        0
signup_date    0
dtype: int64

Unparseable signup_date after coercion: 0
signup_date range: 2023-01-01 00:00:00 to 2024-05-12 00:00:00

Country value counts:
country
Italy      69
Spain      63
Morocco    59
France     56
Germany    53
Name: count, dtype: int64

Country values with leading/trailing whitespace: 0


## 4. `products` — data quality checks

In [17]:
print("Rows:", len(products))
print("Duplicate product_id:", products["product_id"].duplicated().sum())
print("Fully duplicate rows:", products.duplicated().sum())
print("\nNulls per column:\n", products.isnull().sum())
print("\nCategory value counts:")
print(products["category"].value_counts(dropna=False))
print("\nDuplicate product_name (same product listed twice under different ids)?")
print(products[products.duplicated(subset="product_name", keep=False)].sort_values("product_name"))

Rows: 50
Duplicate product_id: 0
Fully duplicate rows: 0

Nulls per column:
 product_id      0
product_name    0
category        0
dtype: int64

Category value counts:
category
Hair      17
Makeup    14
Body      10
Skin       9
Name: count, dtype: int64

Duplicate product_name (same product listed twice under different ids)?
Empty DataFrame
Columns: [product_id, product_name, category]
Index: []


## 5. `orders` — data quality checks

In [18]:
print("Rows:", len(orders))
print("Duplicate order_id:", orders["order_id"].duplicated().sum())
print("Fully duplicate rows:", orders.duplicated().sum())
print("\nNulls per column:\n", orders.isnull().sum())

orders["order_date"] = pd.to_datetime(orders["order_date"], errors="coerce")
print("\nUnparseable order_date after coercion:", orders["order_date"].isnull().sum())
print("order_date range:", orders["order_date"].min(), "to", orders["order_date"].max())

print("\nStatus value counts:")
print(orders["status"].value_counts(dropna=False))

orphan_customer_orders = orders[~orders["customer_id"].isin(customers["customer_id"])]
print("\nOrders referencing a customer_id not in customers.csv:", len(orphan_customer_orders))

Rows: 1000
Duplicate order_id: 0
Fully duplicate rows: 0

Nulls per column:
 order_id       0
customer_id    0
order_date     0
status         0
dtype: int64

Unparseable order_date after coercion: 0
order_date range: 2024-01-01 00:00:00 to 2024-12-30 00:00:00

Status value counts:
status
Completed    805
Cancelled    103
Returned      92
Name: count, dtype: int64

Orders referencing a customer_id not in customers.csv: 0


In [19]:
# Sanity check: can an order predate the customer's own signup date? (would signal bad synthetic
# data or a signup_date that isn't really "first ever interaction")
orders_with_signup = orders.merge(customers[["customer_id", "signup_date"]], on="customer_id", how="left")
before_signup = orders_with_signup[orders_with_signup["order_date"] < orders_with_signup["signup_date"]]
print("Orders placed before the customer's recorded signup_date:", len(before_signup))
if len(before_signup):
    display(before_signup[["order_id", "customer_id", "order_date", "signup_date"]].head())

Orders placed before the customer's recorded signup_date: 55


,order_id,customer_id,order_date,signup_date
79,80,139,2024-02-22,2024-03-08
117,118,231,2024-02-11,2024-02-18
156,157,96,2024-01-11,2024-01-31
165,166,113,2024-02-24,2024-04-12
182,183,247,2024-03-13,2024-05-12


## 6. `order_items` — data quality checks

In [20]:
print("Rows:", len(order_items))
print("Fully duplicate rows:", order_items.duplicated().sum())
print("Duplicate (order_id, product_id) pairs (same product on >1 line of the same order):",
      order_items.duplicated(subset=["order_id", "product_id"]).sum())
print("\nNulls per column:\n", order_items.isnull().sum())

print("\nquantity <= 0:", (order_items["quantity"] <= 0).sum())
print("price <= 0:", (order_items["price"] <= 0).sum())
print("quantity describe:\n", order_items["quantity"].describe())
print("\nprice describe:\n", order_items["price"].describe())

Rows: 2000
Fully duplicate rows: 0
Duplicate (order_id, product_id) pairs (same product on >1 line of the same order): 33

Nulls per column:
 order_id      0
product_id    0
quantity      0
price         0
dtype: int64

quantity <= 0: 0
price <= 0: 0
quantity describe:
 count    2000.000000
mean        2.987500
std         1.407599
min         1.000000
25%         2.000000
50%         3.000000
75%         4.000000
max         5.000000
Name: quantity, dtype: float64

price describe:
 count    2000.000000
mean       64.827000
std        31.191482
min        10.000000
25%        38.000000
50%        66.500000
75%        91.000000
max       119.000000
Name: price, dtype: float64


In [21]:
orphan_order_items = order_items[~order_items["order_id"].isin(orders["order_id"])]
orphan_product_items = order_items[~order_items["product_id"].isin(products["product_id"])]
print("order_items referencing an order_id not in orders.csv:", len(orphan_order_items))
print("order_items referencing a product_id not in products.csv:", len(orphan_product_items))

order_items referencing an order_id not in orders.csv: 0
order_items referencing a product_id not in products.csv: 0


## 7. Cross-table integrity summary

The single most important structural question for this dataset: **does every order have line items, and does every line item belong to a real order?**

In [22]:
orders_with_items = orders["order_id"].isin(order_items["order_id"])
print("Orders WITH at least one line item:", orders_with_items.sum(), f"({orders_with_items.mean():.1%})")
print("Orders WITH NO line items:", (~orders_with_items).sum(), f"({(~orders_with_items).mean():.1%})")

print("\nBreakdown of orders with NO line items, by status:")
print(orders.loc[~orders_with_items, "status"].value_counts())

print("\nBreakdown of ALL orders by status (for comparison):")
print(orders["status"].value_counts())

Orders WITH at least one line item: 861 (86.1%)
Orders WITH NO line items: 139 (13.9%)

Breakdown of orders with NO line items, by status:
status
Completed    110
Cancelled     20
Returned       9
Name: count, dtype: int64

Breakdown of ALL orders by status (for comparison):
status
Completed    805
Cancelled    103
Returned      92
Name: count, dtype: int64


## 8. Cleaning decisions & business rules

Documenting the *why*, not just the *what*, so every downstream number is reproducible and defensible.

| # | Issue found | Decision | Rationale |
|---|---|---|---|
| 1 | `order_date` / `signup_date` loaded as text | Parse to `datetime64` | Required for any monthly trend, cohort, or recency calculation |
| 2 | `orders.status` has `Completed`, `Cancelled`, `Returned` | Define **`is_completed`** flag; only `Completed` orders count as realized revenue in KPIs, cohort, and RFM analysis. `Cancelled` / `Returned` are kept in the cleaned tables and reported separately as an operational metric (cancellation rate, return rate) | Counting cancelled/returned dollars as revenue overstates true business performance — a standard e-commerce convention |
| 3 | Line items exist whose `order_id` doesn't appear in `orders.csv` (if any — see Section 6) | Drop from the fact table, log the count | An order line with no parent order has no customer, date, or status context — it cannot be attributed |
| 4 | Orders exist with **no matching line items** (Section 7) | Excluded from the line-level `fact_sales` table (no product/revenue detail to show); still counted in order-level order/cancellation-rate metrics from `orders.csv` directly | Can't fabricate a revenue amount for an order with no items; excluding silently would understate order volume elsewhere, so it's called out explicitly here instead |
| 5 | Duplicate `(order_id, product_id)` pairs within `order_items` | Kept as separate lines (not summed/deduped) unless the *entire row* is an exact duplicate, in which case the exact duplicate is dropped | A customer can legitimately add the same product to an order in two separate lines (e.g. two different bundle purchases); only a byte-for-byte duplicate row is treated as a data entry error |
| 6 | `country` free text may contain casing/whitespace variants | Strip whitespace, keep original casing (checked above — none found in this dataset, but the step is kept for robustness) | Prevents the same country from being split into two group-by buckets |
| 7 | Revenue is not a stored column in `order_items` | Compute `revenue = quantity * price` at the line level | `price` is unit price; verified against the analyst-provided `sales_cleaned.csv` in Section 9 |


In [23]:
# --- Apply the cleaning decisions ---

customers_clean = customers.copy()
customers_clean["country"] = customers_clean["country"].astype(str).str.strip()
customers_clean = customers_clean.drop_duplicates()

products_clean = products.copy().drop_duplicates()

orders_clean = orders.copy().drop_duplicates()
orders_clean["is_completed"] = orders_clean["status"].eq("Completed")

order_items_clean = order_items.copy()
exact_dupe_lines = order_items_clean.duplicated().sum()
order_items_clean = order_items_clean.drop_duplicates()
order_items_clean = order_items_clean[order_items_clean["order_id"].isin(orders_clean["order_id"])]
order_items_clean = order_items_clean[order_items_clean["product_id"].isin(products_clean["product_id"])]
order_items_clean["revenue"] = order_items_clean["quantity"] * order_items_clean["price"]
order_items_clean.insert(0, "order_item_id", range(1, len(order_items_clean) + 1))

print("Exact duplicate order_item rows dropped:", exact_dupe_lines)
print("order_items_clean shape:", order_items_clean.shape)

Exact duplicate order_item rows dropped: 0
order_items_clean shape: (2000, 6)


## 9. Build the analysis-ready `fact_sales` table

Grain: **one row per order line item**, enriched with customer, product, and order attributes. This is the single source of truth every later notebook (EDA, RFM, cohort, product performance) and the Power BI model will read from.

In [24]:
fact_sales = (
    order_items_clean
    .merge(orders_clean, on="order_id", how="left", validate="many_to_one")
    .merge(customers_clean, on="customer_id", how="left", validate="many_to_one")
    .merge(products_clean, on="product_id", how="left", validate="many_to_one")
)

fact_sales["order_month"] = fact_sales["order_date"].dt.to_period("M").dt.to_timestamp()
fact_sales["order_year"] = fact_sales["order_date"].dt.year

fact_sales = fact_sales[[
    "order_item_id", "order_id", "order_date", "order_month", "order_year", "status", "is_completed",
    "customer_id", "country", "signup_date",
    "product_id", "product_name", "category",
    "quantity", "price", "revenue",
]]

print("fact_sales shape:", fact_sales.shape)
print("\nNulls introduced by the joins (should all be 0):")
print(fact_sales.isnull().sum())
display(fact_sales.head())

fact_sales shape: (2000, 16)

Nulls introduced by the joins (should all be 0):
order_item_id    0
order_id         0
order_date       0
order_month      0
order_year       0
status           0
is_completed     0
customer_id      0
country          0
signup_date      0
product_id       0
product_name     0
category         0
quantity         0
price            0
revenue          0
dtype: int64


,order_item_id,order_id,order_date,order_month,order_year,status,is_completed,customer_id,country,signup_date,product_id,product_name,category,quantity,price,revenue
0,1,508,2024-10-14,2024-10-01,2024,Cancelled,False,170,Morocco,2023-09-07,48,Product_48,Hair,3,108,324
1,2,290,2024-08-26,2024-08-01,2024,Completed,True,256,Germany,2023-06-03,18,Product_18,Makeup,2,93,186
2,3,295,2024-06-25,2024-06-01,2024,Completed,True,54,Morocco,2023-06-12,16,Product_16,Makeup,2,27,54
3,4,451,2024-04-22,2024-04-01,2024,Completed,True,152,Morocco,2023-09-15,39,Product_39,Hair,2,54,108
4,5,904,2024-10-08,2024-10-01,2024,Completed,True,159,France,2023-02-07,4,Product_4,Body,4,117,468


In [25]:
# fact_sales_completed = the subset used for every revenue / RFM / cohort KPI downstream
fact_sales_completed = fact_sales[fact_sales["is_completed"]].reset_index(drop=True)
print("fact_sales_completed shape:", fact_sales_completed.shape)
print("Total realized revenue (Completed only): ${:,.2f}".format(fact_sales_completed["revenue"].sum()))
print("Total revenue across ALL statuses (for reference only): ${:,.2f}".format(fact_sales["revenue"].sum()))

fact_sales_completed shape: (1625, 16)
Total realized revenue (Completed only): $311,111.00
Total revenue across ALL statuses (for reference only): $384,853.00


## 10. Sanity check against the analyst-provided `sales_cleaned.csv`

`sales_cleaned.csv` was supplied alongside the raw tables and appears to already be a Completed-only, joined extract. It is **not** used as an input anywhere above — `fact_sales` is built independently from the four raw tables so the pipeline is fully reproducible. It's used here purely as an external check that our rebuild lands on the same numbers.

In [26]:
sales_reference = pd.read_csv(DATA_RAW / "sales_cleaned.csv")
sales_reference["order_date"] = pd.to_datetime(sales_reference["order_date"], errors="coerce")

print("Reference file (sales_cleaned.csv):")
print("  rows:", len(sales_reference))
print("  distinct order_ids:", sales_reference["order_id"].nunique())
print("  total revenue: ${:,.2f}".format(sales_reference["Revenue"].sum()))

print("\nOur rebuilt fact_sales_completed:")
print("  rows:", len(fact_sales_completed))
print("  distinct order_ids:", fact_sales_completed["order_id"].nunique())
print("  total revenue: ${:,.2f}".format(fact_sales_completed["revenue"].sum()))

print("\nRow-count / order-count differences are expected: sales_cleaned.csv only contains orders\n"
      "that already had line items, while our version starts from the raw order_items table and\n"
      "applies the referential-integrity filters documented in Section 8. Any remaining gap is\n"
      "quantified below rather than assumed away.")

ids_only_in_reference = set(sales_reference["order_id"]) - set(fact_sales_completed["order_id"])
ids_only_in_rebuild = set(fact_sales_completed["order_id"]) - set(sales_reference["order_id"])
print("\nCompleted order_ids present in sales_cleaned.csv but missing from our rebuild:", len(ids_only_in_reference))
print("Completed order_ids present in our rebuild but missing from sales_cleaned.csv:", len(ids_only_in_rebuild))

Reference file (sales_cleaned.csv):
  rows: 1625
  distinct order_ids: 695
  total revenue: $311,111.00

Our rebuilt fact_sales_completed:
  rows: 1625
  distinct order_ids: 695
  total revenue: $311,111.00

Row-count / order-count differences are expected: sales_cleaned.csv only contains orders
that already had line items, while our version starts from the raw order_items table and
applies the referential-integrity filters documented in Section 8. Any remaining gap is
quantified below rather than assumed away.

Completed order_ids present in sales_cleaned.csv but missing from our rebuild: 0
Completed order_ids present in our rebuild but missing from sales_cleaned.csv: 0


## 11. Final validation asserts

Fail loudly, not silently, if a future data refresh breaks an assumption this project relies on.

In [27]:
assert fact_sales["order_item_id"].is_unique, "order_item_id must be unique — fact_sales grain is broken"
assert fact_sales[["order_id", "customer_id", "product_id", "order_date", "revenue"]].isnull().sum().sum() == 0, \
    "Key columns of fact_sales must not contain nulls"
assert (fact_sales["revenue"] >= 0).all(), "Revenue should never be negative"
assert (fact_sales["quantity"] > 0).all(), "Quantity should always be positive"
assert fact_sales_completed["is_completed"].all(), "fact_sales_completed must only contain Completed orders"

print("All validation checks passed.")

All validation checks passed.


## 12. Save cleaned tables to `data/processed/`

These files are the reproducible handoff point: SQL scripts, later notebooks, and the Power BI data model all read from here rather than re-deriving cleaning logic.

In [28]:
customers_clean.to_csv(DATA_PROCESSED / "customers_clean.csv", index=False)
products_clean.to_csv(DATA_PROCESSED / "products_clean.csv", index=False)
orders_clean.to_csv(DATA_PROCESSED / "orders_clean.csv", index=False)
order_items_clean.to_csv(DATA_PROCESSED / "order_items_clean.csv", index=False)
fact_sales.to_csv(DATA_PROCESSED / "fact_sales.csv", index=False)
fact_sales_completed.to_csv(DATA_PROCESSED / "fact_sales_completed.csv", index=False)

print("Saved to", DATA_PROCESSED)
for f in sorted(DATA_PROCESSED.glob("*.csv")):
    print(" -", f.name)

Saved to /Users/yvonne/Desktop/E-commerce Analytics/data/processed
 - customers_clean.csv
 - fact_sales.csv
 - fact_sales_completed.csv
 - order_items_clean.csv
 - orders_clean.csv
 - products_clean.csv


## 13. Data quality summary

A single printed summary of every issue found and how it was resolved — useful as the "data cleaning log" section of the final write-up.

In [29]:
summary = {
    "customers: rows": len(customers),
    "customers: duplicate ids": int(customers["customer_id"].duplicated().sum()),
    "products: rows": len(products),
    "products: duplicate ids": int(products["product_id"].duplicated().sum()),
    "orders: rows": len(orders),
    "orders: Completed / Cancelled / Returned": orders["status"].value_counts().to_dict(),
    "orders: with no line items": int((~orders_with_items).sum()),
    "order_items: raw rows": len(order_items),
    "order_items: exact duplicate rows dropped": int(exact_dupe_lines),
    "order_items: orphaned (bad order_id) dropped": int(len(orphan_order_items)),
    "order_items: orphaned (bad product_id) dropped": int(len(orphan_product_items)),
    "fact_sales: final line-item rows (all statuses)": len(fact_sales),
    "fact_sales_completed: final line-item rows": len(fact_sales_completed),
    "fact_sales_completed: total revenue": round(float(fact_sales_completed["revenue"].sum()), 2),
}
for k, v in summary.items():
    print(f"{k}: {v}")

customers: rows: 300
customers: duplicate ids: 0
products: rows: 50
products: duplicate ids: 0
orders: rows: 1000
orders: Completed / Cancelled / Returned: {'Completed': 805, 'Cancelled': 103, 'Returned': 92}
orders: with no line items: 139
order_items: raw rows: 2000
order_items: exact duplicate rows dropped: 0
order_items: orphaned (bad order_id) dropped: 0
order_items: orphaned (bad product_id) dropped: 0
fact_sales: final line-item rows (all statuses): 2000
fact_sales_completed: final line-item rows: 1625
fact_sales_completed: total revenue: 311111.0


## Next steps

With `data/processed/fact_sales.csv` and `fact_sales_completed.csv` validated and saved, the next notebooks build on top of this table:

- `02_exploratory_data_analysis.ipynb` — revenue trends, MoM growth, order volume, AOV, revenue by category/region
- `03_rfm_segmentation.ipynb` — recency/frequency/monetary scoring and customer segments
- `04_cohort_retention.ipynb` — acquisition-month cohorts and retention matrix
- `05_product_performance.ipynb` — product/category revenue contribution and concentration

The equivalent logic is also implemented in SQL under `sql/` (schema + load scripts for now; analysis queries land alongside their matching notebook).